## NOTE: might need to install all packages from starter code

^^^^^^^

In [ ]:
# # Additional dependencies for SFT
# .venv/bin/pip install trl peft accelerate bitsandbytes datasetsndbytes datasets

In [ ]:
import importlib, sys

packages = [
    "torch", "transformers", "peft", "trl",
    "bitsandbytes", "accelerate", "datasets",
    "vllm", "sympy", "numpy", "tokenizers",
]

print(f"Python: {sys.version}\n" + "-" * 50)
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        print(f"{pkg:<20} {getattr(mod, '__version__', 'unknown')}")
    except ImportError:
        print(f"{pkg:<20} NOT INSTALLED")

# Also check CUDA
try:
    import torch
    print(f"\n{'CUDA available':<20} {torch.cuda.is_available()}")
    print(f"{'CUDA version':<20} {torch.version.cuda}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {props.name}  |  {props.total_memory / 1e9:.1f} GB  |  SM {props.major}.{props.minor}")
except ImportError:
    pass

## Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("open-r1/OpenR1-Math-220k", name="default")
train_ds = ds["train"]

print(f"Loaded {len(train_ds)} examples")

# Remove MCQ examples
train_ds = train_ds.filter(lambda x: x["question_type"] != "MCQ")
print(f"After removing MCQ: {len(train_ds)} examples")

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

def format_example(example):
    solution_with_answer = (
        example["solution"].rstrip()
        + f"\n\nFinal Answer: \\boxed{{{example['answer']}}}"
    )
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": example["problem"]},
            {"role": "assistant", "content": solution_with_answer},
        ]
    }

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,        # nested quantization → saves ~0.4 GB
    bnb_4bit_quant_type="nf4",             # NormalFloat4 — optimal for normally distributed weights
    # bnb_4bit_compute_dtype=torch.float16,  # A30 lacks BF16 tensor cores — use fp16
    bnb_4bit_compute_dtype=torch.bfloat16,  # H100
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False          # Required for gradient checkpointing
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"        # Required for SFT packing

In [ ]:
# print(
#     tokenizer.apply_chat_template(
#         train_ds[0]["messages"],
#         tokenize=False
#     )
# )

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)  # Casts LayerNorm to fp32, enables grad checkpointing

lora_config = LoraConfig(
    r=32,                          # Rank — higher = more capacity, more memory
    lora_alpha=16,                 # Scaling factor: effective LR ≈ lora_alpha / r
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~1–3% of total params trainable

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./checkpoints/sft_qlora",

    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,

    gradient_checkpointing=True,
    optim="paged_adamw_32bit",

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    # fp16=True, # for A30
    bf16=True,   # for H100

    logging_steps=50,
    save_steps=500,
    save_strategy="steps",
    save_total_limit=2,

    report_to="none",

    max_length=2048,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

# trainer.train()
trainer.train(resume_from_checkpoint=True) # if resuming training

trainer.save_model("./models/sft_qlora_adapter")
tokenizer.save_pretrained("./models/sft_qlora_adapter")

In [ ]:
# Must reload base model in fp16/bf16 (not 4-bit) for merging
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    # torch_dtype=torch.float16,   # A30: fp16
    torch_dtype=torch.bfloat16,   # H100
    device_map="auto",
    trust_remote_code=True,
)
peft_model = PeftModel.from_pretrained(base_model, "./models/sft_qlora_adapter")
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained("./models/sft_merged", safe_serialization=True)
tokenizer.save_pretrained("./models/sft_merged")